In [ ]:
!pip -q install -U "transformers>=4.41" accelerate bitsandbytes safetensors
!pip -q install -U diffusers
!pip install -U open_clip_torch

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import time, json, re
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Any

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
# ----------------------------
# Paths
# ----------------------------
BASE = Path("/content/drive/MyDrive/CC_final")
OUT  = BASE / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

def new_session_dir():
    sid = time.strftime("%Y%m%d-%H%M%S")
    sdir = OUT / sid
    sdir.mkdir(parents=True, exist_ok=True)
    return sid, sdir

print("Output root:", OUT)

In [ ]:
# ----------------------------
# LLM
# ----------------------------
LLM_ID = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"

tokenizer = AutoTokenizer.from_pretrained(LLM_ID, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    LLM_ID,
    device_map="auto",
    torch_dtype=torch.float16,
)

model.eval()
print("LLM loaded on:", model.device)

In [ ]:
# ----------------------------
# JSON utils
# ----------------------------
def extract_first_json_block(s: str) -> str:
    start = s.find("{")
    if start == -1:
        raise ValueError("No JSON found")
    depth = 0
    in_string = False
    escaped = False
    for i in range(start, len(s)):
        ch = s[i]
        if in_string:
            if escaped:
                escaped = False
            elif ch == "\\":
                escaped = True
            elif ch == '"':
                in_string = False
            continue
        if ch == '"':
            in_string = True
        elif ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return s[start:i+1]
    raise ValueError("Unbalanced JSON")

def safe_json_loads(text: str) -> Any:
    # Strip code fences if any
    t = text.strip()
    t = re.sub(r"^```(?:json)?\s*", "", t, flags=re.IGNORECASE)
    t = re.sub(r"\s*```$", "", t)
    return json.loads(extract_first_json_block(t))

# ----------------------------
# Token cap utils
# ----------------------------
PROMPT_TOKEN_CAP = 80

def count_tokens(text: str) -> int:
    return len(tokenizer.encode(text, add_special_tokens=False))

def trim_to_tokens(text: str, max_toks: int = PROMPT_TOKEN_CAP) -> str:
    ids = tokenizer.encode(text, add_special_tokens=False)
    ids = ids[:max_toks]
    return tokenizer.decode(ids, skip_special_tokens=True).strip()

# ----------------------------
# LLM chat
# ----------------------------
@torch.inference_mode()
def llm_chat(system_prompt: str, user_prompt: str, max_new_tokens=512, temperature=0.35):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    chat = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(chat, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=0.85,
        repetition_penalty=1.05,
    )
    gen = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

SYSTEM_STRICT_JSON = (
    "You are PromptSmith, a co-creative assistant for crafting text-to-image prompts.\n"
    "Return STRICT JSON only. No extra text.\n"
    "Use SD-style comma-separated visual phrases when writing prompts.\n"
)

SYSTEM_Q_STRICT_JSON = (
    "You are PromptSmith. You generate tailored user questions.\n"
    "Return STRICT JSON only. No extra text, no markdown.\n"
    "Do NOT use double quotes (\") inside any string values.\n"
    "No trailing commas.\n"
)

# ----------------------------
# Schema design
# ----------------------------
KEY_FIELDS = ["colors", "foreground_objects", "background_mood", "style", "camera", "lighting"]
SCHEMA_KEYS = ["subject_scene"] + KEY_FIELDS + ["constraints"]

def _empty_parts() -> Dict[str, str]:
    return {k: "" for k in SCHEMA_KEYS}

NONE_SYNONYMS = {
    "", "none", "none specified", "not specified", "n/a", "na", "null", "unknown", "-"
}

def _norm(s) -> str:
    if s is None:
        return ""
    if not isinstance(s, str):
        s = str(s)
    s = s.strip()
    if s.lower() in NONE_SYNONYMS:
        return ""
    s = re.sub(r"\s+", " ", s).strip()
    return s

def sanitize_parts(parts: Dict[str, str]) -> Dict[str, str]:
    """
    1) remove placeholders (none specified etc.)
    2) keep foreground_objects as "objects besides the main subject"
    3) basic cleanup of commas/spacing
    """
    out = dict(parts)
    for k in SCHEMA_KEYS:
        out[k] = _norm(out.get(k, ""))

    subj = out.get("subject_scene", "").lower()
    fg = out.get("foreground_objects", "")

    if fg:
        for bad in ["cat", "kitten", "dog", "person", "man", "woman", "boy", "girl", "human"]:
            if bad in subj:
                fg = re.sub(rf"\b{bad}\b", "", fg, flags=re.IGNORECASE)

        fg = re.sub(r"\s*,\s*", ", ", fg)
        fg = re.sub(r"(^[,\s]+|[,\s]+$)", "", fg)
        out["foreground_objects"] = fg

    return out

def pretty_schema(parts: Dict[str, str]) -> str:
    p = sanitize_parts(parts)
    lines = []
    for k in SCHEMA_KEYS:
        v = p.get(k, "")
        lines.append(f"- {k}: {v if v else '(empty)'}")
    return "\n".join(lines)

def join_prompt(parts: Dict[str, str]) -> str:
    """
    Natural-ish SDXL prompt, no placeholders, no duplicated subject,
    concise + token capped.
    """
    p = sanitize_parts(parts)

    subject = p.get("subject_scene", "")
    colors  = p.get("colors", "")
    fg      = p.get("foreground_objects", "")
    mood    = p.get("background_mood", "")
    style   = p.get("style", "")
    camera  = p.get("camera", "")
    light   = p.get("lighting", "")
    cons    = p.get("constraints", "")

    clauses = []

    if style:
        if "realistic" in style.lower() and "photo" not in style.lower():
            clauses.append("A realistic photo of")
        else:
            clauses.append(f"A {style} image of")
    else:
        clauses.append("An image of")

    core = subject if subject else "the scene"
    if colors and colors.lower() not in core.lower():
        core = f"{core} ({colors})"
    clauses.append(core)

    if fg:
        clauses.append(f"with {fg} in the foreground")

    if mood:
        clauses.append(f"in a {mood} setting")

    if camera:
        clauses.append(f"Camera: {camera}.")
    if light:
        clauses.append(f"Lighting: {light}.")

    if cons:
        clauses.append(f"{cons}.")

    text = " ".join([c.strip().strip(".") for c in clauses if c.strip()])
    if not text.endswith("."):
        text += "."

    return trim_to_tokens(text, PROMPT_TOKEN_CAP)

In [ ]:
def merge_with_lock(base_parts: Dict[str, str], proposed_parts: Dict[str, str], lock_keys: set) -> Dict[str, str]:
    """
    lock_keys: fields that MUST preserve the user's base value exactly.
    For unlocked keys: proposed can fill freely.
    """
    base = sanitize_parts(base_parts)
    prop = sanitize_parts(proposed_parts)

    out = dict(base)

    bsub = base.get("subject_scene", "")
    psub = prop.get("subject_scene", "")
    if psub and psub.lower() != bsub.lower():
        if bsub:
            out["subject_scene"] = f"{bsub}, {psub}"
        else:
            out["subject_scene"] = psub

    for k in KEY_FIELDS:
        b = base.get(k, "")
        p = prop.get(k, "")

        if k in lock_keys:
            out[k] = b
        else:
            out[k] = p or b

    bc = base.get("constraints", "")
    pc = prop.get("constraints", "")
    if bc and pc and bc.lower() not in pc.lower():
        out["constraints"] = f"{bc}, {pc}"
    else:
        out["constraints"] = pc or bc

    return sanitize_parts(out)

def round1_base(brief: str) -> Dict[str, Any]:
    user = f"""
Summarize the user's brief into ONE concise subject_scene phrase for text-to-image generation.

Return STRICT JSON ONLY:
{{
  "brief_echo": "...(must match the user brief exactly)...",
  "subject_scene": "...(short phrase, must be clearly about the brief)...",
  "assumptions": ["..."],
  "questions_optional": ["..."]
}}

Rules:
- brief_echo MUST equal the user brief exactly (character-for-character).
- subject_scene MUST explicitly include the main subject from the brief.
- Do NOT output placeholders like "none specified".
- No extra keys. No extra text.

User brief: {brief}
"""
    raw = llm_chat(SYSTEM_STRICT_JSON, user, max_new_tokens=300, temperature=0.1)
    try:
        data = safe_json_loads(raw)

        echo = str(data.get("brief_echo", "")).strip()
        subj = _norm(str(data.get("subject_scene", "")))

        if echo != brief or not subj:
            raise ValueError("Round1 JSON failed validation (echo mismatch or empty subject_scene).")

        brief_tokens = {t for t in re.findall(r"[a-zA-Z]+", brief.lower()) if len(t) >= 3}
        subj_tokens  = set(re.findall(r"[a-zA-Z]+", subj.lower()))
        if brief_tokens and len(brief_tokens.intersection(subj_tokens)) == 0:
            raise ValueError("Round1 subject_scene seems unrelated to brief.")

        return {
            "subject_scene": subj,
            "assumptions": data.get("assumptions", []) or [],
            "questions_optional": data.get("questions_optional", []) or [],
        }

    except Exception as e:
        fallback = _norm(brief)
        if not fallback.lower().startswith(("a ", "an ", "the ")):
            fallback = "a " + fallback
        return {
            "subject_scene": fallback,
            "assumptions": [],
            "questions_optional": []
        }

# ---------- Mode A questions ----------
def make_custom_field_questions(brief: str, subject_scene: str) -> Dict[str, Dict[str, Any]]:
    base_user_prompt = f"""
Create tailored questions to collect these fields:
colors, foreground_objects, background_mood, style, camera, lighting.

Requirements:
- Questions MUST be customized to the user's brief (domain-specific wording).
- Provide 2-3 short example answers for each question, also customized.
- Keep each question <= 25 words.
- Keep each example <= 10 words.
- IMPORTANT: Do NOT use any double quotes (") in your output strings.

Return STRICT JSON only:
{{
  "questions": {{
    "colors": {{"q":"...","examples":["...","..."]}},
    "foreground_objects": {{"q":"...","examples":["...","..."]}},
    "background_mood": {{"q":"...","examples":["...","..."]}},
    "style": {{"q":"...","examples":["...","..."]}},
    "camera": {{"q":"...","examples":["...","..."]}},
    "lighting": {{"q":"...","examples":["...","..."]}}
  }}
}}

User brief: {brief}
Interpreted subject/scene: {subject_scene}
""".strip()

    last_raw = None
    for attempt in range(3):
        temp = 0.2 if attempt == 0 else 0.1
        raw = llm_chat(SYSTEM_Q_STRICT_JSON, base_user_prompt, max_new_tokens=650, temperature=temp)
        last_raw = raw
        try:
            data = safe_json_loads(raw)
            qs = data.get("questions", {}) or {}

            out = {}
            for k in KEY_FIELDS:
                item = qs.get(k, {}) or {}
                out[k] = {
                    "q": str(item.get("q", "")).strip() or f"Provide {k}:",
                    "examples": (item.get("examples", []) or [])[:3]
                }
            return out

        except Exception:
            # ask the model to FIX JSON only
            base_user_prompt = f"""
Your previous output was invalid JSON.
Fix it and output ONLY valid JSON (same schema), with no extra text.

Remember:
- No double quotes (") inside values
- No trailing commas

Invalid output:
{last_raw}
""".strip()

    return fallback_field_questions(brief, subject_scene)

def fallback_field_questions(brief: str, subject_scene: str) -> Dict[str, Dict[str, Any]]:
    subj = (subject_scene or brief or "").strip()

    t = (brief + " " + subject_scene).lower()
    is_arch = any(w in t for w in ["house", "building", "room", "interior", "architecture"])
    is_animal = any(w in t for w in ["cat", "dog", "kitten", "puppy", "animal"])

    if is_arch:
        examples_colors = ["white + dark wood", "warm beige", "grey + black trim"]
        examples_fg = ["front garden", "stone pathway", "wooden door"]
        examples_mood = ["calm minimalist", "cozy lived-in", "modern luxury"]
        examples_style = ["photorealistic", "watercolor", "architectural render"]
        examples_cam = ["wide exterior", "street-level 24mm", "interior eye-level"]
        examples_light = ["golden hour", "overcast soft", "night warm lights"]
    elif is_animal:
        examples_colors = ["white + black spots", "soft pastel tones", "warm orange highlights"]
        examples_fg = ["toy ball", "blanket", "food bowl"]
        examples_mood = ["cozy", "playful", "peaceful"]
        examples_style = ["cartoon", "photorealistic", "storybook illustration"]
        examples_cam = ["close-up portrait", "low angle", "rule-of-thirds"]
        examples_light = ["window soft light", "studio softbox", "warm lamp light"]
    else:
        examples_colors = ["warm palette", "cool palette", "high contrast"]
        examples_fg = ["one key prop", "minimal props", "rich props"]
        examples_mood = ["calm", "energetic", "mysterious"]
        examples_style = ["photorealistic", "illustration", "3D render"]
        examples_cam = ["close-up", "medium shot", "wide shot"]
        examples_light = ["soft diffused", "dramatic rim light", "golden hour"]

    return {
        "colors": {"q": f"What color palette fits {subj} best?", "examples": examples_colors[:3]},
        "foreground_objects": {"q": "Any key props to include in the foreground?", "examples": examples_fg[:3]},
        "background_mood": {"q": "What overall mood should the background convey?", "examples": examples_mood[:3]},
        "style": {"q": "Preferred visual style?", "examples": examples_style[:3]},
        "camera": {"q": "How should the scene be framed by the camera?", "examples": examples_cam[:3]},
        "lighting": {"q": "What lighting setup do you want?", "examples": examples_light[:3]},
    }

# ---------- Round 2: build schema, optionally enrich only constraints/empty fields ----------
def build_schema_mode_a(subject_scene: str, user_choices: Dict[str, str]) -> Dict[str, str]:
    parts = _empty_parts()
    parts["subject_scene"] = _norm(subject_scene)
    for k in KEY_FIELDS:
        parts[k] = _norm(user_choices.get(k, ""))
    parts["constraints"] = ""
    return sanitize_parts(parts)

def infer_schema_mode_b(user_text: str, fallback_subject_scene: str) -> Dict[str, str]:
    user = f"""
Infer schema fields from the user's text.

Return STRICT JSON:
{{
  "subject_scene":"...",
  "key_elements": {{
    "colors":"...",
    "foreground_objects":"...",
    "background_mood":"...",
    "style":"...",
    "camera":"...",
    "lighting":"..."
  }},
  "constraints":"..."
}}

Rules:
- Do NOT output placeholders like "none specified".
- foreground_objects must list ONLY objects besides the main subject (do not repeat the subject).
- Keep values short natural phrases.

User text:
{user_text}
"""
    parts = _empty_parts()
    raw = llm_chat(SYSTEM_STRICT_JSON, user, max_new_tokens=650, temperature=0.3)
    try:
        data = safe_json_loads(raw)
    except (ValueError, json.JSONDecodeError):
        parts["subject_scene"] = _norm(fallback_subject_scene or user_text)
        return sanitize_parts(parts)

    parts["subject_scene"] = _norm(str(data.get("subject_scene", fallback_subject_scene)))
    key = data.get("key_elements", {}) or {}
    for k in KEY_FIELDS:
        parts[k] = _norm(str(key.get(k,"")))
    parts["constraints"] = _norm(str(data.get("constraints","")))
    return sanitize_parts(parts)

def enrich_constraints_if_needed(brief: str, parts: Dict[str, str], allow_fill_keys: set) -> Dict[str, str]:
    """
    Optional: ask LLM to fill ONLY allowed empty keys + add constraints.
    This keeps Mode A stable (allow_fill_keys usually empty),
    and lets Mode B fill missing fields reasonably.
    """
    base = sanitize_parts(parts)
    user = f"""
You may improve the schema ONLY in allowed fields if they are empty,
and you may add constraints. Do not change locked content.

Allowed fields to fill (only if currently empty): {sorted(list(allow_fill_keys))}

Return STRICT JSON:
{{"parts": {json.dumps(_empty_parts())} }}

Rules:
- Do NOT output placeholders like "none specified".
- Keep everything concise.

User brief: {brief}
Current schema:
{json.dumps(base, ensure_ascii=False)}
"""
    raw = llm_chat(SYSTEM_STRICT_JSON, user, max_new_tokens=650, temperature=0.35)
    try:
        data = safe_json_loads(raw)
    except (ValueError, json.JSONDecodeError):
        return base
    proposed = data.get("parts", {}) or {}

    prop_parts = _empty_parts()
    for k in SCHEMA_KEYS:
        prop_parts[k] = _norm(str(proposed.get(k, "")))

    # merge: only allow fill on allow_fill_keys; all other key fields locked if base has value
    lock_keys = set()
    for k in KEY_FIELDS:
        if k not in allow_fill_keys:
            lock_keys.add(k)

    merged = merge_with_lock(base, prop_parts, lock_keys=lock_keys)
    return sanitize_parts(merged)


# ---------- Round 3: blueprint-driven, diverse, respects locks ----------
def _concat_unique(base: str, add: str) -> str:
    base = _norm(base)
    add  = _norm(add)
    if not add:
        return base
    if not base:
        return add
    # avoid duplication
    if add.lower() in base.lower():
        return base
    return f"{base}, {add}"

def apply_additions(base_parts: Dict[str, str], additions: Dict[str, str]) -> Dict[str, str]:
    out = sanitize_parts(dict(base_parts))
    additions = additions or {}

    out["subject_scene"] = _concat_unique(out.get("subject_scene",""), additions.get("subject_scene",""))

    for k in KEY_FIELDS:
        out[k] = _concat_unique(out.get(k,""), additions.get(k,""))

    out["constraints"] = _concat_unique(out.get("constraints",""), additions.get("constraints",""))

    return sanitize_parts(out)

GENERIC_TAGS = {"action snap", "cozy lifestyle", "story beat", "action", "cozy", "story", "variant", "option"}

def generate_intent_variants(brief: str, base_parts: Dict[str, str], lock_keys: set, num_variants: int = 3) -> List[Dict[str, Any]]:
    base = sanitize_parts(base_parts)

    user = f"""
You are PromptSmith. Create {num_variants} DISTINCT, customized options for the user's brief.

You MUST respect the base schema.
Locked fields (must remain unchanged):
{sorted(list(lock_keys))}

Your job in this round:
- Propose domain-specific intent_tag titles (NOT generic; avoid "Action snap/Cozy lifestyle/Story beat").
- For each option, output ONLY additive snippets ("additions") for each schema field.
- Additions can be empty strings when not needed.
- Ensure options are clearly different in intent/story AND visual direction.

Diversity requirements (must satisfy):
- Each option must differ in at least 3 of: story detail, motif/prop detail, palette nuance, mood nuance, camera nuance, lighting nuance.
- Use an empty addition for every locked field.
- Keep each addition short (<= 10 words). Natural English phrases, no placeholders.

Return STRICT JSON:
{{
  "variants": [
    {{
      "intent_tag": "...",
      "why": "one short sentence",
      "additions": {{
        "subject_scene": "...",
        "colors": "...",
        "foreground_objects": "...",
        "background_mood": "...",
        "style": "...",
        "camera": "...",
        "lighting": "...",
        "constraints": "..."
      }}
    }}
  ]
}}

User brief: {brief}
Base schema:
{json.dumps(base, ensure_ascii=False)}
"""
    raw = llm_chat(SYSTEM_STRICT_JSON, user, max_new_tokens=900, temperature=0.55)
    try:
        data = safe_json_loads(raw)
    except (ValueError, json.JSONDecodeError):
        data = {}

    variants = []
    seen_prompts = set()

    for i, v in enumerate((data.get("variants", []) or [])[:num_variants]):
        tag = _norm(v.get("intent_tag",""))
        why = _norm(v.get("why",""))

        if not tag or tag.lower() in GENERIC_TAGS:
            tag = f"Option {i+1}"

        additions = v.get("additions", {}) or {}
        add_norm = {k: _norm(additions.get(k, "")) for k in SCHEMA_KEYS}
        for k in lock_keys:
            if k in add_norm:
                add_norm[k] = ""

        merged_parts = apply_additions(base, add_norm)
        prompt = join_prompt(merged_parts)

        if prompt in seen_prompts:
            merged_parts["constraints"] = _concat_unique(merged_parts.get("constraints",""), f"distinct variation {i+1}")
            merged_parts = sanitize_parts(merged_parts)
            prompt = join_prompt(merged_parts)

        seen_prompts.add(prompt)

        variants.append({
            "intent_tag": tag,
            "why": why or "A distinct, brief-specific direction that preserves your choices.",
            "parts": merged_parts,
            "prompt": prompt
        })

    while len(variants) < num_variants:
        merged_parts = dict(base)
        variants.append({
            "intent_tag": f"Option {len(variants)+1}",
            "why": "Fallback variant preserving locked fields.",
            "parts": merged_parts,
            "prompt": join_prompt(merged_parts)
        })

    return variants

In [ ]:
# ----------------------------
# Critic
# ----------------------------
def critic_feedback(brief: str, prompt_text: str):
    user = f"""
You are a brief critic. Evaluate the prompt against the user's brief.
Return STRICT JSON:
{{"alignment":"High/Medium/Low","comment":"one short line","suggestion":"one short line"}}

User brief: {brief}
Current prompt: {prompt_text}
"""
    raw = llm_chat(SYSTEM_STRICT_JSON, user, max_new_tokens=220, temperature=0.25)
    try:
        data = safe_json_loads(raw)
        return (
            str(data.get("alignment","Medium")).strip(),
            str(data.get("comment","")).strip(),
            str(data.get("suggestion","")).strip()
        )
    except Exception:
        return "Medium", "Critic fallback.", "Try making style/camera/lighting more explicit."

# ----------------------------
# Refine per module
# ----------------------------
def refine_schema_part(brief: str, parts: Dict[str, str], target_key: str, feedback: str) -> (Dict[str, str], str):
    assert target_key in SCHEMA_KEYS
    user = f"""
Refine ONLY the selected schema field based on user feedback.
Keep other fields unchanged.
The updated value should be short and SD-style.

Return STRICT JSON:
{{"updated_value":"...","change_summary":"one short sentence"}}

User brief: {brief}
Current schema: {json.dumps(parts, ensure_ascii=False)}
Selected field: {target_key}
User feedback: {feedback}
"""
    raw = llm_chat(SYSTEM_STRICT_JSON, user, max_new_tokens=320, temperature=0.35)
    new_parts = dict(parts)
    try:
        data = safe_json_loads(raw)
        upd = str(data.get("updated_value","")).strip()
        if upd:
            new_parts[target_key] = upd
        return new_parts, (str(data.get("change_summary","")).strip() or "Updated.")
    except Exception:
        return new_parts, "No change: the model response could not be parsed."

# ----------------------------
# CLIP
# ----------------------------
import open_clip
from PIL import Image
from IPython.display import display

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="openai"
)
clip_tokenizer = open_clip.get_tokenizer("ViT-B-32")
clip_model = clip_model.to("cuda").eval()

@torch.inference_mode()
def clip_text_image_score(text: str, image: Image.Image) -> float:
    img = clip_preprocess(image).unsqueeze(0).to("cuda")
    txt = clip_tokenizer([text]).to("cuda")
    img_feat = clip_model.encode_image(img)
    txt_feat = clip_model.encode_text(txt)
    img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)
    txt_feat = txt_feat / txt_feat.norm(dim=-1, keepdim=True)
    return float((img_feat @ txt_feat.T).squeeze().item())

def trim_clip_text(text: str, max_toks: int = 77) -> str:
    ids = clip_tokenizer([text])
    if ids.shape[1] <= max_toks:
        return text
    words = text.split()
    while words:
        candidate = " ".join(words)
        ids = clip_tokenizer([candidate])
        if ids.shape[1] <= max_toks:
            return candidate
        words = words[:-1]
    return text[:128]

In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline

torch.backends.cuda.matmul.allow_tf32 = True

SDXL_ID = "stabilityai/stable-diffusion-xl-base-1.0"

dtype = torch.float16

pipe = StableDiffusionXLPipeline.from_pretrained(
    SDXL_ID,
    dtype=dtype,
    use_safetensors=True,
    variant="fp16",
)

# pipe.enable_xformers_memory_efficient_attention()
pipe.vae.enable_tiling()
pipe.enable_model_cpu_offload()

print("Loaded:", SDXL_ID)

def generate_image_sdxl(
    prompt: str,
    negative_prompt: str = "",
    seed: int = 0,
    steps: int = 30,
    guidance: float = 6.0,
    height: int = 1024,
    width: int = 1024,
) -> Image.Image:
    g = torch.Generator(device="cpu").manual_seed(int(seed))
    img = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=int(steps),
        guidance_scale=float(guidance),
        height=int(height),
        width=int(width),
        generator=g,
    ).images[0]
    return img

In [ ]:
# ----------------------------
# State + Main loop (Mode A/B)
# ----------------------------
def ask_int(prompt: str, default: int, minimum: int = 0) -> int:
    raw = input(prompt).strip()
    try:
        value = int(raw) if raw else default
    except ValueError:
        print(f"Invalid integer; using {default}.")
        return default
    return max(minimum, value)

def ask_float(prompt: str, default: float, minimum: float = 0.0) -> float:
    raw = input(prompt).strip()
    try:
        value = float(raw) if raw else default
    except ValueError:
        print(f"Invalid number; using {default}.")
        return default
    return max(minimum, value)

def ask_size(prompt: str, default: str = "1024x1024") -> tuple[int, int]:
    raw = input(prompt).strip().lower() or default
    try:
        h, w = (int(x) for x in raw.split("x", 1))
        if h <= 0 or w <= 0 or h % 8 or w % 8:
            raise ValueError
        return h, w
    except ValueError:
        print(f"Invalid size; using {default}.")
        return tuple(int(x) for x in default.split("x"))

@dataclass
class SessionState:
    mode: str = "A"
    brief: str = ""
    extra_freeform: str = ""
    prompt_base: str = ""
    subject_scene: str = ""
    parts: Dict[str, str] = field(default_factory=_empty_parts)
    variants: List[Dict[str, Any]] = field(default_factory=list)
    chosen_variant_idx: Optional[int] = None
    history: List[Dict[str, Any]] = field(default_factory=list)
    refine_rounds: int = 0
    start_time: float = 0.0
    last_clip: Optional[float] = None
    clip_text: str = ""

def run_promptsmith():
    sid, sdir = new_session_dir()
    state = SessionState()
    state.start_time = time.time()

    print("="*68)
    print("PromptSmith (Colab CLI) — type 'quit' to exit")
    print("Session:", sid)
    print("="*68)

    mode = input("\nChoose mode: (A) Structured / (B) Freeform : ").strip().upper()
    if mode not in {"A","B"}:
        mode = "A"
    state.mode = mode

    brief = input("\n[1] Enter your rough image idea:\n> ").strip()
    if brief.lower() in {"quit","exit"}:
        return
    state.brief = brief
    state.history.append({"stage":"brief", "mode": mode, "brief": brief})

    if mode == "B":
        state.extra_freeform = input("\n[2] Add any extra details (optional, free text):\n> ").strip()
        state.history.append({"stage":"freeform_details", "extra": state.extra_freeform})

    print("\n[Round 1] Base prompt")
    r1 = round1_base(state.brief if mode == "A" else (state.brief + " " + state.extra_freeform).strip())
    state.subject_scene = r1["subject_scene"]
    print("subject_scene:", state.subject_scene)

    if r1.get("assumptions"):
        print("\nAssumptions:")
        for a in r1["assumptions"][:5]:
            print("-", a)

    if r1.get("questions_optional"):
        print("\nOptional questions (not required):")
        for q in r1["questions_optional"][:4]:
            print("-", q)

    # Mode A: tailored questions to collect the 6 key fields
    user_choices = {k:"" for k in KEY_FIELDS}

    # Mode A/B schema construction
    user_choices = {k: "" for k in KEY_FIELDS}

    if mode == "A":
        print("\n[Mode A] Tailored questions (mapped to 6 key fields)")
        qpack = make_custom_field_questions(state.brief, state.subject_scene)

        for k in KEY_FIELDS:
            q = qpack[k]["q"]
            ex = qpack[k]["examples"]
            print(f"\n{k.upper()} — {q}")
            if ex:
                print("  examples:", " | ".join(ex))
            ans = input("> ").strip()
            user_choices[k] = ans

        state.history.append({"stage": "collect_key_fields", "user_choices": dict(user_choices)})

        state.parts = build_schema_mode_a(state.subject_scene, user_choices)
        print("\n[Schema] (Mode A, user-locked):")
        print(pretty_schema(state.parts))
        print("\nPROMPT_SCHEMA:", join_prompt(state.parts))

        state.history.append({"stage": "schema_mode_a", "parts": dict(state.parts)})

        lock_keys = set(KEY_FIELDS)

    else:
        print("\n[Mode B] Inferring key fields from your free text")
        user_text = (state.brief + " " + state.extra_freeform).strip()

        state.parts = infer_schema_mode_b(
            user_text=user_text,
            fallback_subject_scene=state.subject_scene
        )

        allow_fill = {k for k in KEY_FIELDS if not state.parts.get(k)}
        state.parts = enrich_constraints_if_needed(
            brief=state.brief,
            parts=state.parts,
            allow_fill_keys=allow_fill
        )

        print("\n[Schema] (Mode B, inferred):")
        print(pretty_schema(state.parts))
        print("\nPROMPT_SCHEMA:", join_prompt(state.parts))

        state.history.append({"stage": "schema_mode_b", "parts": dict(state.parts)})

        lock_keys = {k for k in KEY_FIELDS if state.parts.get(k)}

    print("\n[Round 3] Intent-based variants (3 options)")
    state.variants = generate_intent_variants(
        brief=state.brief,
        base_parts=state.parts,
        lock_keys=lock_keys,
        num_variants=3
    )

    for i, v in enumerate(state.variants):
        print(f"\n[{i}] {v['intent_tag']}")
        print("why:", v["why"])
        print("prompt:", v["prompt"])

    while True:
        pick = input("\nChoose variant index (0/1/2): ").strip()
        if pick.isdigit() and int(pick) in range(3):
            pick = int(pick)
            break
        print("Invalid. Try again.")

    state.chosen_variant_idx = pick
    state.parts = dict(state.variants[pick]["parts"])
    prompt_now = join_prompt(state.parts)
    state.history.append({"stage":"choose_variant", "index": pick, "intent_tag": state.variants[pick]["intent_tag"], "prompt": prompt_now})

    # Critic after choosing
    a, cmt, sug = critic_feedback(state.brief, prompt_now)
    print("\n[Critic] alignment:", a)
    print("[Critic] comment  :", cmt)
    print("[Critic] suggest  :", sug)
    state.history.append({"stage":"critic", "alignment": a, "comment": cmt, "suggestion": sug})

    state.clip_text = state.brief
    if state.extra_freeform:
        state.clip_text = f"{state.brief}. {state.extra_freeform}"
    state.clip_text = trim_clip_text(state.clip_text, max_toks=77)

    gen_now = input("\nGenerate an image preview now? (y/n): ").strip().lower()
    if gen_now == "y":
        seed = ask_int("Seed (int, default 0): ", 0)
        steps = ask_int("Steps (default ~30): ", 30, minimum=1)
        guidance = ask_float("Guidance (default 6.0): ", 6.0)
        h, w = ask_size("Size HxW (default 1024x1024): ")

        img = generate_image_sdxl(prompt_now, seed=seed, steps=steps, guidance=guidance, height=h, width=w)
        display(img)
        p = str(sdir / f"preview_seed{seed}_steps{steps}_g{guidance}_{h}x{w}.png")
        img.save(p)
        clip_score = clip_text_image_score(state.clip_text, img)
        state.last_clip = clip_score
        print("Saved:", p)
        print("CLIP(text, image) =", clip_score)
        state.history.append({"stage":"image", "seed": seed, "steps": steps, "guidance": guidance, "size": [h,w], "path": p, "clip": clip_score})

    print("\n[Refinement loop]")
    print("Commands:")
    print("  - 'refine' : refine one schema field")
    print("  - 'img'    : generate image + CLIP score")
    print("  - 'show'   : show current schema and prompt")
    print("  - 'done'   : finalize\n")

    while True:
        cmd = input("Cmd> ").strip().lower()
        if cmd in {"done","finish"}:
            break

        if cmd == "show":
            print("\nCurrent SCHEMA:")
            print(pretty_schema(state.parts))
            print("\nPROMPT:", join_prompt(state.parts))
            continue

        if cmd == "img":
            prompt_now = join_prompt(state.parts)

            seed = ask_int("Seed (int, default 0): ", 0)
            steps = ask_int("Steps (default 30): ", 30, minimum=1)
            guidance = ask_float("Guidance (default 6.0): ", 6.0)
            h, w = ask_size("Size HxW (default 1024x1024): ")

            img = generate_image_sdxl(
                prompt=prompt_now,
                seed=seed,
                steps=steps,
                guidance=guidance,
                height=h,
                width=w
            )

            display(img)

            p = str(sdir / f"sdxl_seed{seed}_steps{steps}_g{guidance}_{h}x{w}.png")
            img.save(p)

            clip_score = clip_text_image_score(state.clip_text, img)
            state.last_clip = clip_score

            print("Saved:", p)
            print("CLIP(text, image) =", clip_score)

            state.history.append({
                "stage":"image",
                "engine":"sdxl_base_1.0",
                "seed": seed,
                "steps": steps,
                "guidance": guidance,
                "size": [h, w],
                "path": p,
                "clip": clip_score
            })
            continue

        if cmd == "refine":
            print("\nWhich field to refine?")
            for i, k in enumerate(SCHEMA_KEYS):
                print(f"  [{i}] {k}")
            idx = input("Select index: ").strip()
            if not (idx.isdigit() and int(idx) in range(len(SCHEMA_KEYS))):
                print("Invalid selection.")
                continue
            target_key = SCHEMA_KEYS[int(idx)]
            feedback = input(f"Feedback for '{target_key}': ").strip()
            if not feedback:
                print("Empty feedback, skipped.")
                continue

            before = dict(state.parts)
            state.parts, summary = refine_schema_part(state.brief, state.parts, target_key, feedback)
            state.refine_rounds += 1

            prompt_now = join_prompt(state.parts)
            print("\nChange:", summary)
            print("Updated field:", target_key)
            print("PROMPT:", prompt_now)

            # Critic after refinement
            a, cmt, sug = critic_feedback(state.brief, prompt_now)
            print("\n[Critic] alignment:", a)
            print("[Critic] comment  :", cmt)
            print("[Critic] suggest  :", sug)

            state.history.append({
                "stage":"refine",
                "round": state.refine_rounds,
                "target_key": target_key,
                "feedback": feedback,
                "change_summary": summary,
                "before": before,
                "after": dict(state.parts),
                "critic": {"alignment": a, "comment": cmt, "suggestion": sug}
            })
            continue

        print("Unknown command. Use: refine / img / show / done")

    # Save session
    elapsed = time.time() - state.start_time
    final_prompt = join_prompt(state.parts)
    metrics = {
        "mode": state.mode,
        "refine_rounds": state.refine_rounds,
        "elapsed_seconds": elapsed,
        "last_clip_score": state.last_clip,
        "prompt_tokens": count_tokens(final_prompt),
        "clip_text": state.clip_text,
        "img_model": SDXL_ID,
        "llm_model": LLM_ID,
    }

    final = {
        "session_id": sid,
        "final_schema": state.parts,
        "final_prompt": final_prompt,
        "history": state.history,
        "metrics": metrics
    }

    out_json = sdir / "session.json"
    out_json.write_text(json.dumps(final, ensure_ascii=False, indent=2), encoding="utf-8")
    print("\nSaved session log:", out_json)
    print("\nFINAL PROMPT:\n", final_prompt)
    print("\nMETRICS:\n", metrics)

run_promptsmith()